In [2]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
from torch.utils.data import DataLoader

In [3]:
# Check if GPU is enabled
print(torch.cuda.is_available())  # should print True
print(torch.cuda.get_device_name(0))  # should print the GPU name

True
Tesla T4


In [4]:
# Check GPU memory (to check if 512 batch size is viable)
print(torch.cuda.get_device_name(0))
print(f"Memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Tesla T4
Memory available: 15.6 GB


In [5]:
# Mount Google Drive so model checkpoints can be saved
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
# Define augmentation pipeline according to SimCLR paper

transform = transforms.Compose([
    transforms.RandomResizedCrop(size=32, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [8]:
# Define a wrapper class essentially so augmentation is applied twice -> results in two independent views per image.

class TwoViewDataset:
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        img, label = self.base_dataset[index]
        view1 = self.transform(img)
        view2 = self.transform(img)
        return (view1, view2), label

In [9]:
# Load base CIFAR10
base_cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=None)

# Wrap it -> create trainset and its DataLoader!
trainset = TwoViewDataset(base_dataset=base_cifar10, transform=transform)
trainloader = DataLoader(trainset, batch_size=512, shuffle=True, num_workers=2)

100%|██████████| 170M/170M [00:04<00:00, 42.5MB/s]


In [10]:
import torch.nn as nn

In [11]:
# First load the ResNet-18 classifier
resnet18 = torchvision.models.resnet18(weights=None, progress=True)

# Reduce kernel size and stride as CIFAR10 images are very small
resnet18.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)

# Remove max pooling layer for same reason
resnet18.maxpool = nn.Identity()

# Remove final classification layer (as embeddings are in penultimate layer)
resnet18.fc = nn.Identity()

In [12]:
# Define projection head (only used during training)
class ProjectionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(512, 512)
        self.bn = nn.BatchNorm1d(512)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(512, 128)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.fc2(x)
        return F.normalize(x, dim=1)

In [13]:
# Define SimCLR model (a wrapper of the ResNet-18 and projection head)

class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = resnet18
        self.projection = ProjectionHead()

    def forward(self, x1, x2):
        h1 = self.encoder(x1)  # (batch_size, 512)
        h2 = self.encoder(x2)  # (batch_size, 512)

        z1 = self.projection(h1)  # (batch_size, 128)
        z2 = self.projection(h2)  # (batch_size, 128)

        return z1, z2

    # Use after training for embedding extraction
    def get_embedding(self, x):
        return self.encoder(x)


In [14]:
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, z1, z2):
      out = torch.cat([z1, z2], dim=0)
      n_samples = len(out)

      # Full similarity matrix
      cov = torch.mm(out, out.t().contiguous())
      sim = torch.exp(cov / self.temperature)

      mask = ~torch.eye(n_samples, device=sim.device).bool()
      neg = sim.masked_select(mask).view(n_samples, -1).sum(dim=-1)

      # Positive similarity
      pos = torch.exp(torch.sum(z1 * z2, dim=-1) / self.temperature)
      pos = torch.cat([pos, pos], dim=0)

      loss = -torch.log(pos / neg).mean()
      return loss

In [15]:
# Instantiate requirements for training loop

model = SimCLR().to(device)
num_epochs = 500
optimiser = torch.optim.SGD(model.parameters(), lr=0.4, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=num_epochs)
criterion = NTXentLoss(temperature=0.5).to(device)

In [16]:
# Continue model training

checkpoint_path = '/content/drive/MyDrive/5CCSAMLF_CW2/models/simclr_latest.pth'

checkpoint = torch.load(checkpoint_path)

model.load_state_dict(checkpoint['model_state_dict'])
optimiser.load_state_dict(checkpoint['optimiser_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_epoch = checkpoint['epoch'] + 1  # resume from next epoch

print(f"Resuming from epoch {start_epoch}, last loss: {checkpoint['loss']:.4f}")


Resuming from epoch 350, last loss: 5.1689


In [17]:
# Train the model

checkpoint_path = '/content/drive/MyDrive/5CCSAMLF_CW2/models/simclr_latest.pth'

model.train()
for epoch in range(start_epoch, num_epochs):
    total_loss = 0

    for (x1, x2), labels in trainloader:
        x1, x2 = x1.to(device), x2.to(device)

        optimiser.zero_grad()

        z1, z2 = model(x1, x2)
        loss = criterion(z1, z2)

        loss.backward()
        optimiser.step()

        total_loss += loss.item()

    scheduler.step()

    avg_loss = total_loss / len(trainloader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f}")

    # Save checkpoint periodically
    if (epoch + 1) % 50 == 0:
      torch.save({'epoch': epoch,
                  'model_state_dict': model.state_dict(),
                  'optimiser_state_dict': optimiser.state_dict(),
                  'scheduler_state_dict': scheduler.state_dict(),
                  'loss': avg_loss},
                  checkpoint_path)
      print(f"Model checkpoint at epoch {epoch+1}")

Epoch [351/500] Loss: 5.1650
Epoch [352/500] Loss: 5.1645
Epoch [353/500] Loss: 5.1646
Epoch [354/500] Loss: 5.1643
Epoch [355/500] Loss: 5.1628
Epoch [356/500] Loss: 5.1629
Epoch [357/500] Loss: 5.1611
Epoch [358/500] Loss: 5.1610
Epoch [359/500] Loss: 5.1574
Epoch [360/500] Loss: 5.1609
Epoch [361/500] Loss: 5.1603
Epoch [362/500] Loss: 5.1596
Epoch [363/500] Loss: 5.1579
Epoch [364/500] Loss: 5.1597
Epoch [365/500] Loss: 5.1580
Epoch [366/500] Loss: 5.1580
Epoch [367/500] Loss: 5.1589
Epoch [368/500] Loss: 5.1551
Epoch [369/500] Loss: 5.1544
Epoch [370/500] Loss: 5.1538
Epoch [371/500] Loss: 5.1541
Epoch [372/500] Loss: 5.1525
Epoch [373/500] Loss: 5.1547
Epoch [374/500] Loss: 5.1526
Epoch [375/500] Loss: 5.1516
Epoch [376/500] Loss: 5.1515
Epoch [377/500] Loss: 5.1515
Epoch [378/500] Loss: 5.1523
Epoch [379/500] Loss: 5.1494
Epoch [380/500] Loss: 5.1476
Epoch [381/500] Loss: 5.1503
Epoch [382/500] Loss: 5.1482
Epoch [383/500] Loss: 5.1473
Epoch [384/500] Loss: 5.1485
Epoch [385/500